In [1]:
import pandas as pd
# On recharge le fichier CSV généré à l'étape précédente
df = pd.read_csv("churn_data.csv")
df.head() # affiche les 5 premières lignes, pour un contrôle visuel rapide

,age,salaire,abonnement,anciennete,consommation,churn
0,22,1973,Premium,7,87,0
1,58,5633,Premium,3,6,0
2,52,7745,Standard,3,39,0
3,40,7098,Premium,4,45,0
4,40,3063,Basique,8,79,0


In [2]:
df.shape # (nombre de lignes, nombre de colonnes) -> doit afficher (500, 6)

(500, 6)

In [4]:
df.describe() # count (500 clients), moyenne, min, max, écart-type, 25%, 50%, 75% des clients

,age,salaire,anciennete,consommation,churn
count,500.000000,500.000000,500.000000,500.00000,500.000000
mean,43.456000,4645.940000,6.772000,48.65600,0.046000
std,14.858318,1982.304927,4.377023,28.32586,0.209695
min,18.000000,1229.000000,0.000000,0.00000,0.000000
25%,31.000000,2870.250000,3.000000,25.00000,0.000000
50%,43.000000,4759.000000,7.000000,46.00000,0.000000
75%,56.250000,6416.750000,11.000000,73.25000,0.000000
max,69.000000,7993.000000,14.000000,99.00000,1.000000


In [5]:
df.isnull().sum() # compte les valeurs manquantes par colonne -> les 500 lignes sont complètes, aucune donnée n'est vide, doit être 0 partout

age             0
salaire         0
abonnement      0
anciennete      0
consommation    0
churn           0
dtype: int64

In [6]:
df["churn"].value_counts() # compte les clients "0" (restent) vs "1" (partent)

churn
0    477
1     23
Name: count, dtype: int64

In [9]:
from sklearn.model_selection import train_test_split
# ============================================================
# 1. Conversion de la colonne « abonnement »
# ============================================================
# Les algorithmes de ML ne comprennent que des nombres : on convertit
# le texte de la colonne "abonnement" en valeurs numériques.
df["abonnement"] = df["abonnement"].map({
"Basique": 0,
"Standard": 1,
"Premium": 2
})
# ============================================================
# 2. Définition des variables explicatives (features)
# ============================================================
X = df[
[
"age",
"salaire",
"abonnement",
"anciennete",
"consommation"
]
]
# ============================================================
# 3. Définition de la variable cible (target)
# ============================================================
y = df["churn"]
# ============================================================
# 4. Séparation Train / Test
# - 80% pour entraîner le modèle (X_train, y_train)
# - 20% mis de côté, jamais vus pendant l'entraînement,
# pour évaluer honnêtement le modèle (X_test, y_test)
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.2, # 20% des données réservées au test
random_state=42 # même logique de reproductibilité que pour le dataset
)

In [10]:
X_train.shape

(400, 5)

In [11]:
X_test.shape

(100, 5)

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
# On crée le modèle : une forêt de 100 arbres de décision qui votent ensemble.
# max_depth=5 limite la complexité de chaque arbre (évite qu'il "apprenne par cœur").
model = RandomForestClassifier(
    n_estimators=100,
max_depth=5,
random_state=42
)
# .fit() est le mot qui déclenche l'APPRENTISSAGE proprement dit :
# le modèle analyse X_train et y_train pour trouver une relation entre les deux.
# La cellule qui affiche « RandomForestClassifier(...) » indique simplement
# que le modèle a été entraîné avec succès. À ce stade, aucune prédiction
# n'a encore été réalisée : le modèle a appris à partir des 400 clients
# d'entraînement et est maintenant prêt à être testé sur les 100 restants.
model.fit(X_train, y_train)
# On demande au modèle de prédire sur les données de test (jamais vues avant).
y_pred = model.predict(X_test)
# accuracy_score compare les prédictions (y_pred) à la vérité (y_test)
# et renvoie le pourcentage de bonnes réponses.
print("Accuracy :", accuracy_score(y_test, y_pred))
# classification_report détaille la précision et le rappel, classe par classe.
print(classification_report(y_test, y_pred))

Accuracy : 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        94
           1       1.00      1.00      1.00         6

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



In [13]:
import joblib
# joblib.dump() sérialise (transforme en fichier binaire) l'objet "model" entraîné.
joblib.dump(model, "model.pkl")
print("Modèle sauvegardé dans model.pkl")

Modèle sauvegardé dans model.pkl


In [14]:
# ---------- ESSAI 2 : un modèle plus simple ----------
model = RandomForestClassifier(
n_estimators=50, # moins d'arbres qu'à l'essai 1 (100 -> 50)
max_depth=3, # arbres moins profonds qu'à l'essai 1 (5 -> 3)
random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy Essai 2 :", accuracy_score(y_test, y_pred))
joblib.dump(model, "model.pkl") # écrase le fichier de l'Essai 1
print("model.pkl écrasé avec l'Essai 2")

Accuracy Essai 2 : 1.0
model.pkl écrasé avec l'Essai 2


In [19]:

# ---------- ESSAI 3 : on retire la colonne "salaire" ----------

X = df[["age", "abonnement", "anciennete", "consommation"]]
# La colonne "salaire" est volontairement retirée

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(
    n_estimators=100, max_depth=5, random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy Essai 3 (sans salaire) :", accuracy_score(y_test, y_pred))

joblib.dump(model, "model.pkl")  # Écrase le fichier de l'Essai 2

print("model.pkl écrasé avec l'Essai 3")


Accuracy Essai 3 (sans salaire) : 1.0
model.pkl écrasé avec l'Essai 3


In [20]:
# ---------- Vérification : que contient VRAIMENT model.pkl maintenant ? ----------
loaded = joblib.load("model.pkl")
# n_features_in_ indique combien de colonnes ce modèle attend en entrée :
# 5 -> c'est l'Essai 1 ou 2 ; 4 -> c'est forcément l'Essai 3 (sans salaire)
print("Nombre de features attendu :", loaded.n_features_in_)

Nombre de features attendu : 4
